In [ ]:
import json
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
from prettytable import PrettyTable

from src.utils.config import Config
from src.utils.helpers import init_this_notebook, p


config = Config()
init_this_notebook(config.SEED)

# Mapping of zip files to their extraction targets
paths = { config.PATHS_TRAIN_IMAGES_ZIP: config.PATHS_TRAIN_IMAGES,
          config.PATHS_EVAL_IMAGES_ZIP: config.PATHS_EVAL_IMAGES }

p("paths", paths)

config.JSON_PATH = config.PATHS_DATA / "train_annotations.json"



In [ ]:
# Check current device
device = "cuda" if torch.cuda.is_available() else torch.device('cpu')
pin_memory = True if device == "cuda" else False
p("Device", f"Running on {device} with pin_memory: {pin_memory} ")

In [ ]:

# Bounding box from polygon
def bbox_segmentation( segmentation ):
    """
    Convert a segmentation polygon (flat or list of lists)
    into a [x_min, y_min, x_max, y_max] bounding box.
    """
    xs, ys = [], []

    # Flatten any nested polygons
    if isinstance(segmentation[0], list):
        # xs, ys = [], []
        for seg in segmentation:
            xs.extend(seg[0::2])
            ys.extend(seg[1::2])
    else:
        xs = segmentation[0::2]
        ys = segmentation[1::2]

    if not xs or not ys:
        return [0, 0, 0, 0]

    return [min(xs), min(ys), max(xs), max(ys)]


def bbox_segmentation_individual( segmentation ):
    """
    Convert segmentation(s) to one or more [x_min, y_min, x_max, y_max] bounding boxes.
    Returns a list of boxes (even if only one).
    """
    if not segmentation:
        return []

    boxes = []
    if isinstance(segmentation[0], list):
        for seg in segmentation:
            xs = seg[0::2]
            ys = seg[1::2]
            boxes.append([min(xs), min(ys), max(xs), max(ys)])
    else:
        xs = segmentation[0::2]
        ys = segmentation[1::2]
        boxes.append([min(xs), min(ys), max(xs), max(ys)])

    return boxes


def bbox_segmentation_group( segmentation, include_group_box = True ):
    """
    Convert a segmentation polygon (or list of polygons)
    into a list of bounding boxes.
    Returns a dict with 'individual_boxes' and optional 'group_box'.
    """
    if not segmentation:
        return { "individual_boxes": [], "group_box": None }

    if isinstance(segmentation[0], list):
        # multiple polygons
        all_x, all_y = [], []
        individual_boxes = []
        for seg in segmentation:
            xs = seg[0::2]
            ys = seg[1::2]
            individual_boxes.append([min(xs), min(ys), max(xs), max(ys)])
            all_x.extend(xs)
            all_y.extend(ys)
        group_box = [min(all_x), min(all_y), max(all_x), max(all_y)] if include_group_box else None
    else:
        xs = segmentation[0::2]
        ys = segmentation[1::2]
        individual_boxes = [[min(xs), min(ys), max(xs), max(ys)]]
        group_box = individual_boxes[0] if include_group_box else None

    return { "individual_boxes": individual_boxes, "group_box": group_box }
    # if not segmentation:
    #     return []
    #
    # all_x, all_y = [], []
    # boxes = []
    #
    # if isinstance(segmentation[0], list):
    #     # multiple polygons
    #     for seg in segmentation:
    #         xs = seg[0::2]
    #         ys = seg[1::2]
    #         boxes.append({
    #                 "type": "individual",
    #                 "bbox": [min(xs), min(ys), max(xs), max(ys)]
    #         })
    #         all_x.extend(xs)
    #         all_y.extend(ys)
    #     if include_group_box:
    #         boxes.append({
    #                 "type": "group",
    #                 "bbox": [min(all_x), min(all_y), max(all_x), max(all_y)]
    #         })
    # else:
    #     # single polygon
    #     xs = segmentation[0::2]
    #     ys = segmentation[1::2]
    #     boxes.append({
    #             "type": "individual",
    #             "bbox": [min(xs), min(ys), max(xs), max(ys)]
    #     })
    #     if include_group_box:
    #         boxes.append({
    #                 "type": "group",
    #                 "bbox": [min(xs), min(ys), max(xs), max(ys)]
    #         })
    #
    # return boxes


# Load annotations (now mirrors the working single-cell behavior)
def load_annotations( json_path ):
    with open(json_path, "r") as f:
        data = json.load(f)

    all_items = []
    for item in data.get("images", []):
        img_path = Path(item["file_name"])
        #width, height = item.get("width"), item.get("height")
        annotations = item.get("annotations", [])
        bboxes = []

        for ann in annotations:
            clss = ann.get("class", "unknown")
            segmentation = ann.get("segmentation", [])
            confidence = ann.get("confidence_score", 1.0)

            # if not segmentation or len(segmentation) < 4:
            #     continue

            # Each annotation is a flat polygon → one bbox
            bbox = bbox_segmentation(segmentation)
            bboxes.append({
                    "class": clss,
                    "bbox": bbox,
                    "confidence_score": confidence
            })
            # boxes = bbox_segmentation_individual(segmentation)
            # boxes = bbox_segmentation_group(segmentation)
            #
            # # Add individual boxes
            # for box in boxes["individual_boxes"]:
            #     bboxes.append({
            #             "class": clss,
            #             "bbox": box,
            #             "bbox_type": "individual",
            #             "confidence_score": confidence
            #     })
            # # Add group box (optional)
            # if boxes["group_box"] is not None:
            #     bboxes.append({
            #             "class": clss,
            #             "bbox": boxes["group_box"],
            #             "bbox_type": "group",
            #             "confidence_score": confidence
            #     })

        all_items.append({ "image_path": img_path, "bboxes": bboxes })
    return all_items


# Unique classes
def get_unique_classes( annotations ):
    classes = { bbox["class"] for item in annotations for bbox in item["bboxes"] }
    return sorted(classes)


# Visualization helper
def show_image_with_mask( image_path, mask_path, bboxes = None, alpha = 0.3, plot_bboxes = True ):
    image = cv2.imread(str(image_path))
    if image is None:
        p("Failed to read image", image_path)
        return
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
    if mask is None:
        p("Failed to read mask", mask_path)
        return

    mask_rgb = np.zeros_like(image)
    mask_rgb[:, :, 0] = mask
    overlay = cv2.addWeighted(image, 1 - alpha, mask_rgb, alpha, 0)

    if plot_bboxes and bboxes:
        for bbox in bboxes:
            # box = bbox["bbox"]
            # if isinstance(box[0], list):  # nested
            #     box = box[0]
            box = bbox.get("bbox")
            if not isinstance(box, (list, tuple)) or len(box) != 4:
                continue  # skip bad data
            x1, y1, x2, y2 = map(int, box)
            #x1, y1, x2, y2 = map(int, bbox["bbox"])

            if bbox["class"] == "individual_tree":
                color = (0, 255, 0)  # green
            elif bbox["class"] == "group_of_trees":
                #color = (255, 255, 0)  # yellow
                color = (255, 255, 0)  #if bbox.get("bbox_type") == "individual" else (255, 128, 0)
            else:
                color = (255, 0, 0)  # blue (fallback)
            #
            # color = (0, 255, 0) if bbox["class"] == "individual_tree" \
            #     else (0, 0, 255) if bbox["class"] == "group_of_trees" \
            #     else (255, 0, 0)
            #

            cv2.rectangle(overlay, (x1, y1), (x2, y2), color, 1)
            #label = bbox["class"]
            label = str(1 if bbox["class"] == "individual_tree" else 2 if bbox["class"] == "group_of_trees" else 0)
            #label = f"{bbox['class'][0].upper()}-{bbox.get('bbox_type', '')[:1].upper()}"

            org = (x1, max(10, y1 - 5))
            fontFace = cv2.FONT_HERSHEY_SIMPLEX
            fontScale = 0.4
            thickness = 1

            cv2.putText(
                overlay,  # image to draw on
                label,  # text string
                org,  # (x, y) coordinates for bottom-left corner
                fontFace,  # cv2.FONT_HERSHEY_SIMPLEX
                fontScale,  # text size multiplier
                color,  # (B, G, R)
                thickness,  # line thickness
                #lineType=cv2.LINE_AA    # (optional) anti-aliasing
            )

    plt.figure(figsize = (4, 4), dpi = 200)
    # plt.subplot(1, 2, 1)
    # plt.imshow(image)
    # plt.title("Image")
    # plt.axis("off")

    #plt.subplot(1, 2, 2)
    plt.imshow(overlay)
    plt.title("With Mask + BBoxes")
    plt.axis("off")
    plt.tight_layout()
    plt.show()


def find_images_with_both_classes( annotations, class_a = "individual_tree", class_b = "group_of_trees" ):
    """
    Return a list of image paths that contain both class_a and class_b annotations.
    """
    results = []

    for item in annotations:
        classes_in_image = { bbox["class"] for bbox in item["bboxes"] }
        if class_a in classes_in_image and class_b in classes_in_image:
            results.append(item["image_path"])

    return results


def show_annotations_for_image( image_name, annotations, config, alpha = 0.4 ):
    """
    Display bounding boxes and mask for a specific image.
    image_name: e.g. '30.tif'
    """

    # find the image entry
    item = next((a for a in annotations if a["image_path"].name == image_name), None)
    if item is None:
        print(f"Image '{image_name}' not found in annotations.")
        return

    img_name = item["image_path"].name
    bboxes = item["bboxes"]

    print(f"\nImage: {img_name}")
    print(f"Total bounding boxes: {len(bboxes)}")

    # summarize class counts
    class_counts = { }
    for b in bboxes:
        class_counts[b["class"]] = class_counts.get(b["class"], 0) + 1

    p("Class counts", class_counts)

    # create a table of all bounding boxes
    table = PrettyTable()
    table.field_names = ["#", "Class", "BBox", "Confidence"]
    for i_bbox, bbox in enumerate(bboxes):
        table.add_row([
                i_bbox + 1,
                bbox["class"],
                bbox["bbox"],
                bbox["confidence_score"]
        ])
    print(table)

    # locate mask and image
    sample_img = config.PATHS_TRAIN_IMAGES / img_name
    sample_mask = config.PATHS_TRAIN_MASKS / img_name

    # visualize with overlay
    show_image_with_mask(sample_img, sample_mask, bboxes = bboxes, alpha = alpha, plot_bboxes = True)






In [ ]:




annotations = load_annotations(config.JSON_PATH)
unique_classes = get_unique_classes(annotations)

images_with_both = find_images_with_both_classes(annotations)
for img_path in images_with_both[:1]:
    p("(sample) ⚠ Mixed-class image", img_path.name, color = "yellow")

p("Unique classes", unique_classes)
images = sorted(config.PATHS_TRAIN_IMAGES.glob("*"))

show_annotations_for_image("10cm_train_14.tif", annotations, config)

for clss in unique_classes:
    p(f"Class: {clss}", color = "red")

    # Sort by how many boxes of this class exist per image
    sorted_annotations = sorted(
        enumerate(annotations),
        key = lambda x: len([b for b in x[1]["bboxes"] if b["class"] == clss]),
        reverse = True
    )

    # Pick the image with the most of this class
    for idx, item in sorted_annotations:
        bboxes = [b for b in item["bboxes"] if b["class"] == clss]
        if not bboxes:
            continue

        img_name = item["image_path"].name
        p(img_name, f"{len(bboxes)} bounding boxes")

        # Call the refactored function here
        show_annotations_for_image(img_name, annotations, config, alpha = 0.4)
        break



#
# for clss in unique_classes:
#     p(f"Class: {clss}")
#
#     # Sort by how many boxes of this class exist per image
#     sorted_annotations = sorted(
#         enumerate(annotations),
#         key = lambda x: len([b for b in x[1]["bboxes"] if b["class"] == clss]),
#         reverse = True
#     )
#
#     # Pick the image with the most of this class
#     for idx, item in sorted_annotations:
#         bboxes = [b for b in item["bboxes"] if b["class"] == clss]
#         if not bboxes:
#             continue
#
#         img_name = item["image_path"].name
#         p(img_name, f"{len(bboxes)} bounding boxes")
#
#         table = PrettyTable()
#         table.field_names = ["#", "Class", "BBox", "Confidence"]
#         for i_bbox, bbox in enumerate(bboxes[:5]):  # limit to first few for display
#             table.add_row([i_bbox + 1, bbox["class"], bbox["bbox"], bbox["confidence_score"]])
#         p(table)
#
#         sample_img = config.PATHS_TRAIN_IMAGES / img_name
#         sample_mask = config.PATHS_TRAIN_MASKS / img_name
#         show_image_with_mask(sample_img, sample_mask, bboxes = bboxes, alpha = 0.4, plot_bboxes = True)
#         break

In [ ]:



target_image = "10cm_train_14.tif"

# 1. Read JSON
with open(config.JSON_PATH, "r") as f:
    data = json.load(f)

# 2. Find the target image entry
image_entry = next((img for img in data["images"] if Path(img["file_name"]).name == target_image), None)
if image_entry is None:
    raise ValueError(f"{target_image} not found in JSON.")

width, height = image_entry["width"], image_entry["height"]

# 3. Create mask (same logic as your create_masks_from_custom_json)
mask = np.zeros((height, width), dtype = np.uint8)
for ann in image_entry["annotations"]:
    segmentation = ann.get("segmentation", [])
    if not segmentation or len(segmentation) < 4:
        continue
    poly = np.array(segmentation).reshape(-1, 2).astype(np.int32)
    cv2.fillPoly(mask, [poly], color = 1)

# 4. Compute bounding boxes (one per annotation)
bboxes = []
for ann in image_entry["annotations"]:
    segmentation = ann.get("segmentation", [])
    if not segmentation or len(segmentation) < 4:
        continue
    xs = segmentation[0::2]
    ys = segmentation[1::2]
    bbox = [min(xs), min(ys), max(xs), max(ys)]
    bboxes.append(bbox)

# 5. Read image
image_path = config.PATHS_TRAIN_IMAGES / target_image
image = cv2.imread(str(image_path))
if image is None:
    raise FileNotFoundError(f"Could not read {image_path}")
image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

# Prepare mask RGB
mask_rgb = np.zeros_like(image)
mask_rgb[:, :, 0] = mask * 255
overlay = cv2.addWeighted(image, 0.7, mask_rgb, 0.3, 0)

# Draw bounding boxes
for (x1, y1, x2, y2) in bboxes:
    cv2.rectangle(overlay, (int(x1), int(y1)), (int(x2), int(y2)), (0, 255, 0), 1)

# 6. Plot
plt.figure(figsize = (15, 5), dpi = 200)
plt.subplot(1, 3, 1)
plt.imshow(image)
plt.title("Image")
plt.axis("off")

plt.subplot(1, 3, 2)
plt.imshow(mask, cmap = "gray")
plt.title("Mask")
plt.axis("off")

plt.subplot(1, 3, 3)
plt.imshow(overlay)
plt.title("Image + Mask + Boxes")
plt.axis("off")

plt.tight_layout()
plt.show()

